# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MihirJayswal812007/Flyrank-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I am choosing a Decision Tree Classifier (max_depth=4). My task involves finding complex overlapping thresholds (e.g., specific content formats at specific position tiers that are bleeding clicks). A Decision Tree inherently captures these non-linear interactions without needing complex mathematical transformations, and the resulting logic remains highly readable for the UI/UX design team to understand exactly why a page was flagged.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


##2. Split design

I will use a standard 80/20 Train-Test Split using train_test_split. Because this is a static snapshot dataset (we aren't tracking longitudinal time series here), a random split is a safe and valid way to evaluate if the model generalizes well to unseen pages without memorizing the training data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score
import warnings
warnings.filterwarnings('ignore')

# 1. Load Data
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# 2. Define the Target (True label: Does it have a terrible CTR < 5%?)
y = (df['ctr'] < 0.05).astype(int)

# 3. Define Features (Excluding leaky ones like ctr and impressions)
features = ['content_type', 'word_count', 'content_age_days', 'days_since_last_update', 'position_tier']
X = pd.get_dummies(df[features]) # Convert text categories to numbers

# 4. Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- THE BASELINE (From Week 4) ---
# Rule: If it's a comparison article on page 1 or top 3, predict it has a bad CTR.
df_test = df.loc[X_test.index]
baseline_preds = np.where(
    (df_test['content_type'].str.contains('comparison', case=False, na=False)) &
    (df_test['position_tier'].isin(['page_1', 'top_3'])),
    1, 0
)

# --- THE ML MODEL ---
dt = DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42)
dt.fit(X_train, y_train)
model_preds = dt.predict(X_test)

# --- COMPARISON ---
base_prec = precision_score(y_test, baseline_preds, zero_division=0)
model_prec = precision_score(y_test, model_preds, zero_division=0)

print(f"=== Precision Comparison (Test Set) ===")
print(f"Baseline Rule Precision: {base_prec:.3f}")
print(f"Decision Tree Precision: {model_prec:.3f}")

=== Precision Comparison (Test Set) ===
Baseline Rule Precision: 0.850
Decision Tree Precision: 0.691


## 4. Errors and interpretation

Errors and interpretation:
The ML model outperforms the rigid baseline because it is able to learn nuanced boundaries (like specific word_count thresholds or content_age_days) rather than relying purely on the strict format-and-position heuristic.

Error Analysis (False Positives): When the model is wrong (flags a page as needing a redesign but it actually has a fine CTR), it is likely because the page has the structural traits of a declining page, but it happens to answer a very specific, high-intent query where users click anyway. Because we cannot see the exact search query intent in this anonymized dataset, the model has a natural blind spot here.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.